# Clase 112 — Transfer learning, unsupervised pretraining

Reusar un modelo preentrenado (ImageNet), reemplazar la cabeza, **congelar** la base, fine-tunear la cabeza y luego **descongelar** con LR muy bajo. Patrón dominante en producción con pocos datos.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. GPU recomendada.

## 1. Cargar una base preentrenada y congelarla

`include_top=False` descarta la cabeza de clasificación de ImageNet; `trainable=False` la congela.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

base = keras.applications.MobileNetV3Small(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False
print("capas de la base:", len(base.layers), "| base.trainable:", base.trainable)

## 2. Cabeza custom sobre la base congelada

`preprocess_input` lleva las imágenes al rango que espera el modelo; `GlobalAveragePooling2D` + `Dropout` + `Dense` forman la nueva cabeza.

In [ ]:
num_clases = 5
inp = keras.Input(shape=(224, 224, 3))
x = keras.applications.mobilenet_v3.preprocess_input(inp)     # normaliza a [-1, 1]
x = base(x, training=False)                                   # BN en modo inference
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
salida = layers.Dense(num_clases, activation="softmax")(x)
modelo = keras.Model(inp, salida)

entrenables = int(sum(np.prod(w.shape) for w in modelo.trainable_weights))
print("params totales:", modelo.count_params(), "| entrenables (solo la cabeza):", entrenables)

## 3. Etapa 1 — entrenar solo la cabeza

Base congelada + `Adam(1e-3)`. Rápido, porque casi ningún peso se actualiza.

In [ ]:
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# train_ds = keras.utils.image_dataset_from_directory(
#     "datos/", image_size=(224, 224), batch_size=32)
# modelo.fit(train_ds, epochs=10)
print("Etapa 1 lista: solo la cabeza entrena con Adam(1e-3).")

## 4. Etapa 2 — fine-tuning con LR bajo

Descongelar la base y **recompilar** con LR 100× más bajo para evitar catastrophic forgetting.

In [ ]:
base.trainable = True                                        # descongelar todo
modelo.compile(optimizer=keras.optimizers.Adam(1e-5),        # LR 100x más bajo
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# modelo.fit(train_ds, epochs=10)
print("Etapa 2 lista: fine-tuning con LR 1e-5 (recompilado tras tocar trainable).")

## 5. Congelar parcialmente y comparar con entrenar desde cero

Congelar las primeras capas (features genéricas) reduce overfitting. Entrenar la misma arquitectura con `weights=None` es el baseline sin transfer.

In [ ]:
for capa in base.layers[:100]:                               # congelar las primeras 100
    capa.trainable = False
entrenables = sum(c.trainable for c in base.layers)
print("capas entrenables tras congelar las primeras 100:", entrenables)

desde_cero = keras.applications.MobileNetV3Small(
    weights=None, include_top=False, input_shape=(224, 224, 3))
print("baseline sin transfer: mismo grafo, sin pesos ImageNet → necesita mucho más dato")

## Ejercicios

1. **Carga y congelado**: cargá `MobileNetV3Small(include_top=False)` y ponela en `trainable=False`.
2. **Modelo completo**: base + `GlobalAveragePooling2D` + `Dropout(0.2)` + `Dense(softmax)`.
3. **Dos etapas**: entrená la cabeza con `Adam(1e-3)`; luego descongelá y recompilá con `Adam(1e-5)`.
4. **Sin transfer**: entrená la misma arquitectura con `weights=None` y compará; con dataset chico, difícilmente iguale al transfer.

## Conclusiones

- Las capas tempranas aprenden features generales (bordes, texturas); reusarlas ahorra datos y cómputo.
- Pipeline estándar: **load → freeze → new head → fit → unfreeze → fit con LR bajo**.
- Cambiar `trainable` **requiere recompilar** el modelo.
- LR bajo en fine-tuning evita **catastrophic forgetting**.
- Para texto, el análogo es cargar BERT/DistilBERT desde Hugging Face y fine-tunear una cabeza.